In [1]:
#!/usr/bin/env python
# coding: utf-8

import os
import subprocess
import concurrent.futures
from joblib import Parallel, delayed

# latencies: 50, 90 150, 210
default_region = ['us-west1-b']
# regions = ['us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
regions = ['us-west1-b', 'us-west1-b', 'us-west1-b', 'us-west1-b']


In [ ]:
# Regions

# num_nodes = 16
zone_no = 0
for num_nodes in  [48,32,16,8]:


    project = "ucr-ursa-major-lesani-lab"
    zone = "us-central1-c"
    machine_type = "e2-highmem-2"
    image_family = "tsm-sc-family"  # your custom image
    subnet = "default"
    gcp_username = "tejas"

    # Fetch all tsm-sc-* instances across ALL zones
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''
    
    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []
    
    for line in output.splitlines():
        if line.strip():
            name, zone = line.split()
            instances.append((name, zone))
    
    print("\n➡ Existing instances to delete:")
    for name, zone in instances:
        print(f"  - {name} ({zone})")
    
    def delete_instance(name, zone):
        cmd = f'''
        gcloud compute instances delete {name} \
            --zone={zone} \
            --project={project} \
            --quiet
        '''
        print(f"🗑️ Deleting {name} in {zone}")
        return subprocess.call(cmd, shell=True)
    
    if instances:
        with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
            futures = [
                executor.submit(delete_instance, name, zone)
                for name, zone in instances
            ]
            concurrent.futures.wait(futures)
    
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")

    
    # Create commands list
    commands = []
    
    for i in range(num_nodes):

        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]
        cmd = f'''
        gcloud compute instances create tsm-sc-{i:03} \
            --project={project} \
            --zone={zone} \
            --machine-type={machine_type} \
            --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
            --can-ip-forward \
            --maintenance-policy=MIGRATE \
            --provisioning-model=STANDARD \
            --service-account=961693926925-compute@developer.gserviceaccount.com \
            --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
            --tags=http-server,https-server \
            --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
            --no-shielded-secure-boot \
            --shielded-vtpm \
            --shielded-integrity-monitoring \
            --labels=goog-ec-src=vm_add-gcloud \
            --reservation-affinity=any
        '''
        commands.append(cmd.strip())
    
    
    def run_command(command):
        print(f"Running: {command}")
        return subprocess.call(command, shell=True)
    
    
    # #Parallel instance creation
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=48) as executor:
        futures = [executor.submit(run_command, cmd) for cmd in commands]
        concurrent.futures.wait(futures)
    
    print("All instances launched.")
    

    # Wait a bit for IPs to propagate
    import time
    time.sleep(30)
    

    # Get IPs
    os.system('gcloud compute instances list --filter="name~\'tsm-sc-\'" '
              '--format="value(networkInterfaces[0].networkIP)" > tsm_ips.txt')
    
    with open('tsm_ips.txt', 'r') as f:
        iplist = [line.strip() for line in f.readlines()]
    
    print("🎯 Instance IPs:", iplist)
    

    os.system('git add .; git commit -m "testing"; git push')
   

    n_collection = 100
    os.system('make -j8')
    
    

    def kill_stellar_private(i):


        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]
        
        remote_command = f"""\
    cd /home/tejas/stellar-private; \
    sudo pkill -9 stellar-core; \
    """
        
        # Construct the full gcloud command
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "{remote_command}"'
        
        print(f"Executing: {command}")
    
        output = os.system(command)
        print(f"Return code for tsm-sc-{i:03}: {output}")
    
    
    results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in range(num_nodes))
    
    

    
    
    
    def git_pull_stellar(i):
        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]
        
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
    cd stellar-core; \
    git pull"'
        print(command)
        output = os.system(command)
        print(output)
    
    # Execute in parallel like your example
    results = Parallel(n_jobs=48)(delayed(git_pull_stellar)(i) for i in range(len(iplist)))
    print(results)
    

    import shutil
    
    if os.path.exists('../stellar-private'):
        
        shutil.rmtree('../stellar-private')
    os.mkdir('../stellar-private')
    
    
    os.system('cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh')
    
    os.system('cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; ./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh')
    
    # --- Configuration ---
    line_to_add = "SEND_CUSTOM_MESSAGE=true"
    target_file = "../stellar-private/node1/stellar-core.cfg" 
    
    # --- The os.system() Command ---
    # This command prepends the line to the target_file on your local machine.
    os.system(f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}')
    
    print(f"The line '{line_to_add}' has been prepended to {target_file}.")
    

    def compile_stellar(i):

        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
    cd stellar-core; \
    make -j16; cd; sudo rm -r stellar-private"'
        print(command)
        output = os.system(command)
        print(output)
    
    # Execute in parallel like your example
    results = Parallel(n_jobs=48)(delayed(compile_stellar)(i) for i in range(len(iplist)))
    print(results)
    

    def clean_stellar_private(i):

        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]

        
        remote_command = f"""\
    cd /home/tejas; \
    sudo rm -r stellar-private; \
    """
        
        # Construct the full gcloud command
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "{remote_command}"'
        
        print(f"Executing: {command}")
    
        output = os.system(command)
        print(f"Return code for tsm-sc-{i:03}: {output}")
    
    
    results = Parallel(n_jobs=20)(delayed(clean_stellar_private)(i) for i in range(num_nodes))
    

    def copy_folder_to_instance(i, source_folder = '/home/tejas/stellar-private',destination_path = '/home/tejas/stellar-private' ):
        """
        Constructs and executes the gcloud compute scp command to copy a folder
        to a specific GCP instance.
        """


        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]
        
        instance_name = f"tsm-sc-{i:03}"
        
        # The --recurse flag is crucial for copying folders
        # The format is: gcloud compute scp --recurse [LOCAL_SRC] [USER]@[INSTANCE_NAME]:[REMOTE_DEST]
        command = f'gcloud compute scp --zone "{zone}" --project "{project}" \
    --recurse "{source_folder}" "{instance_name}:{destination_path}"'
    
        print(f"Executing command for {instance_name}: {command}")
        
        # os.system executes the command and returns the exit status (0 for success)
        output = os.system(command)
        
        print(f"Command for {instance_name} finished with exit code: {output}")
        
        return (instance_name, output)
    
    
    results = Parallel(n_jobs=48)(
        delayed(copy_folder_to_instance)(i) for i in range(num_nodes)
    )
    
    

    
    # def setup_stellar_private(i):
    #     command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
    # cd /home/tejas; \
    # cd stellar-private; chmod +x gcp_setup_stellar_private.sh; \
    # ./gcp_setup_stellar_private.sh;"'
    #     print(command)
    #     output = os.system(command)
    #     print(output)
    
    # # Execute in parallel like your example
    # results = Parallel(n_jobs=20)(delayed(setup_stellar_private)(i) for i in range(len(iplist)))
    # print(results)
    
    def run_stellar_private(i):


        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]
        # Calculate the node number (assuming i starts at 0, node starts at 1)
        node_number = i + 1 
        instance_name = f"tsm-sc-{i:03}"
        
        # ----------------------------------------------------------------------------------
        # FIX: Use nohup, redirect I/O to a log file, and add ' & disown'
        # '2>&1' redirects stderr to stdout. '> log.txt' redirects stdout to a file.
        # '< /dev/null' ensures the process doesn't wait for input.
        # ----------------------------------------------------------------------------------
        remote_command = f"""\
    cd /home/tejas/stellar-private; \
    nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
    > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
    """
        
        # Construct the full gcloud command
        command = f'gcloud compute ssh --zone "{zone}" "{instance_name}" --project "{project}" --command "{remote_command}"'
        
        print(f"Executing: {command}")
        
        # os.system should now return immediately because the remote shell exits
        output = os.system(command)
        print(f"Return code for {instance_name}: {output}")


        
    results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in range(num_nodes))

    # Corrected Loop (to run 0, 1, 2, 3)
    results = Parallel(n_jobs=48)(delayed(run_stellar_private)(i) for i in range(num_nodes))
    # results = Parallel(n_jobs=20)(delayed(run_stellar_private)(i) for i in [3,2,1,0])
    

    # time.sleep(3)
    # for i in range(num_nodes):
    
        # run_stellar_private(num_nodes-i-1)
        # time.sleep(2)
    # run_stellar_private(0)
    print(results)
    print("All SSH commands executed. Nodes should be starting up in the background.")
    
    time.sleep(200)
    

    
    
    
    results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in range(num_nodes))
    
    

    remote_base_folder = "/home/tejas/stellar-private" # The base path on the GCP instance
    # local_base_destination = "/home/tejas/work/experiments/stellar-core/" + "collection_"+str(n_collection)+"_rounds_" + str(num_nodes) + "_zone_" + str(zone_no) 
    local_base_destination = "/home/tejas/work/experiments/stellar-core/" + "scp_" + str(num_nodes) + "_v2"
    
    # Ensure the local base destination directory exists
    os.makedirs(local_base_destination, exist_ok=True)
    
    
    def copy_folder_from_instance(i):

        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]
            
        """
        Constructs and executes the gcloud compute scp command to copy a specific 
        nodeN folder from instance i to a local folder named after the instance.
        """
        instance_name = f"tsm-sc-{i:03}"
        
        # Calculate the node number (assuming i starts at 0, node starts at 1)
        node_number = i + 1 
        node_folder = f"node{node_number}"
    
        # 1. Define the specific REMOTE source path on the instance
        # Example: /home/tejas/stellar-private/node1
        remote_source_path = os.path.join(remote_base_folder, node_folder)
        
        # 2. Define the LOCAL destination path
        # We'll use the instance name for the subfolder to keep backups separate
        local_destination_path = os.path.join(local_base_destination, instance_name)
        os.makedirs(local_destination_path, exist_ok=True)
        
        # The SCp command requires the remote path to be formatted as:
        # [INSTANCE_NAME]:[REMOTE_SRC]
        remote_source = f"{instance_name}:{remote_source_path}"
        
        # The command reverses the source (remote) and destination (local)
        command = f'gcloud compute scp --zone "{zone}" --project "{project}" \
    --recurse "{remote_source}" "{local_destination_path}"'
    
        print(f"Executing command to copy {node_folder} from {instance_name}: {command}")
        
        # os.system executes the command and returns the exit status (0 for success)
        output = os.system(command)
        
        print(f"Copy from {instance_name} finished with exit code: {output}")
        
        return (instance_name, output)
    
    # ---
    # Execute the copy operation in parallel
    # ---
    
    results = Parallel(n_jobs=48)(
        delayed(copy_folder_from_instance)(i) for i in range(3)
    )
    
    print("\n--- Summary of Download Results ---")
    print(results)
    


➡ Existing instances to delete:
  - tsm-sc-000 (us-west1-b)
  - tsm-sc-001 (us-west1-b)
  - tsm-sc-002 (us-west1-b)
  - tsm-sc-003 (us-west1-b)
🗑️ Deleting tsm-sc-000 in us-west1-b
🗑️ Deleting tsm-sc-001 in us-west1-b
🗑️ Deleting tsm-sc-002 in us-west1-b
🗑️ Deleting tsm-sc-003 in us-west1-b
gcloud compute ssh --zone "us-west1-b" "tsm-sc-003" --project "ucr-ursa-major-lesani-lab" --command "    cd stellar-core;     make -j16; cd; sudo rm -r stellar-private"
0
Executing: gcloud compute ssh --zone "us-west1-b" "tsm-sc-003" --project "ucr-ursa-major-lesani-lab" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return code for tsm-sc-003: 0
gcloud compute ssh --zone "us-west1-b" "tsm-sc-001" --project "ucr-ursa-major-lesani-lab" --command "    cd stellar-core;     make -j16; cd; sudo rm -r stellar-private"
0
Executing: gcloud compute ssh --zone "us-west1-b" "tsm-sc-002" --project "ucr-ursa-major-lesani-lab" --command "    cd /home/tejas/stellar-private;  

Exception ignored in: <function ResourceTracker.__del__ at 0x75fd4818a020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x72022278a020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ 

gcloud compute ssh --zone "us-west1-b" "tsm-sc-000" --project "ucr-ursa-major-lesani-lab" --command "    cd stellar-core;     make -j16; cd; sudo rm -r stellar-private"
0
Executing: gcloud compute ssh --zone "us-west1-b" "tsm-sc-001" --project "ucr-ursa-major-lesani-lab" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return code for tsm-sc-001: 0


Exception ignored in: <function ResourceTracker.__del__ at 0x738d5e382020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-002].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-003].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-000].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-001].



🧹 All tsm-sc-* instances deleted across all regions.

Running: gcloud compute instances create tsm-sc-000             --project=ucr-ursa-major-lesani-lab             --zone=us-west1-b             --machine-type=e2-highmem-2             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default             --can-ip-forward             --maintenance-policy=MIGRATE             --provisioning-model=STANDARD             --service-account=961693926925-compute@developer.gserviceaccount.com             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append             --tags=http-server,https-server             --create-disk=auto-delete=yes,boot=yes,image-family=tsm-sc-family,mode=rw,size=20,type=pd-balanced             --no-

...

Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-002].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-021].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-026].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-015].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-014].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-029].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-007].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-011].
Deleted [https://www.goo

🗑️ Deleting tsm-sc-039 in us-west1-b
🗑️ Deleting tsm-sc-040 in us-west1-b
🗑️ Deleting tsm-sc-041 in us-west1-b
🗑️ Deleting tsm-sc-042 in us-west1-b


Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-005].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-003].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-000].


🗑️ Deleting tsm-sc-043 in us-west1-b
🗑️ Deleting tsm-sc-044 in us-west1-b
🗑️ Deleting tsm-sc-045 in us-west1-b
🗑️ Deleting tsm-sc-046 in us-west1-b
🗑️ Deleting tsm-sc-047 in us-west1-b


Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-008].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-030].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-009].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-012].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-028].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-017].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-020].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-025].


In [8]:

    # # Fetch all tsm-sc-* instances across ALL zones
    # fetch_cmd = f'''
    # gcloud compute instances list \
    #     --project={project} \
    #     --filter="name~'^tsm-sc-'" \
    #     --format="value(name,zone)"
    # '''
    
    # output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    # instances = []
    
    # for line in output.splitlines():
    #     if line.strip():
    #         name, zone = line.split()
    #         instances.append((name, zone))
    
    # print("\n➡ Existing instances to delete:")
    # for name, zone in instances:
    #     print(f"  - {name} ({zone})")
    
    # def delete_instance(name, zone):
    #     cmd = f'''
    #     gcloud compute instances delete {name} \
    #         --zone={zone} \
    #         --project={project} \
    #         --quiet
    #     '''
    #     print(f"🗑️ Deleting {name} in {zone}")
    #     return subprocess.call(cmd, shell=True)
    
    # if instances:
    #     with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
    #         futures = [
    #             executor.submit(delete_instance, name, zone)
    #             for name, zone in instances
    #         ]
    #         concurrent.futures.wait(futures)
    
    #     print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    # else:
    #     print("\n✔ No tsm-sc-* instances found.\n")



➡ Existing instances to delete:
  - tsm-sc-000 (us-west1-b)
  - tsm-sc-001 (us-west1-b)
  - tsm-sc-002 (us-west1-b)
  - tsm-sc-003 (us-west1-b)
  - tsm-sc-004 (us-west1-b)
  - tsm-sc-005 (us-west1-b)
  - tsm-sc-006 (us-west1-b)
  - tsm-sc-007 (us-west1-b)
  - tsm-sc-008 (asia-south1-c)
  - tsm-sc-009 (asia-south1-c)
  - tsm-sc-010 (asia-south1-c)
  - tsm-sc-011 (asia-south1-c)
  - tsm-sc-012 (asia-south1-c)
  - tsm-sc-013 (asia-south1-c)
  - tsm-sc-014 (asia-south1-c)
  - tsm-sc-015 (asia-south1-c)
🗑️ Deleting tsm-sc-000 in us-west1-b
🗑️ Deleting tsm-sc-001 in us-west1-b
🗑️ Deleting tsm-sc-002 in us-west1-b
🗑️ Deleting tsm-sc-003 in us-west1-b
🗑️ Deleting tsm-sc-004 in us-west1-b
🗑️ Deleting tsm-sc-005 in us-west1-b
🗑️ Deleting tsm-sc-006 in us-west1-b
🗑️ Deleting tsm-sc-007 in us-west1-b
🗑️ Deleting tsm-sc-008 in asia-south1-c
🗑️ Deleting tsm-sc-009 in asia-south1-c
🗑️ Deleting tsm-sc-010 in asia-south1-c
🗑️ Deleting tsm-sc-011 in asia-south1-c
🗑️ Deleting tsm-sc-012 in asia-south1-c

Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/asia-south1-c/instances/tsm-sc-010].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/asia-south1-c/instances/tsm-sc-012].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/asia-south1-c/instances/tsm-sc-014].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/asia-south1-c/instances/tsm-sc-008].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/asia-south1-c/instances/tsm-sc-011].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/asia-south1-c/instances/tsm-sc-009].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/asia-south1-c/instances/tsm-sc-013].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/asia-south1-c/instances/tsm-sc-015].



🧹 All tsm-sc-* instances deleted across all regions.



Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-west1-b/instances/tsm-sc-006].
Exception ignored in: <function ResourceTracker.__del__ at 0x780e84996020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7ceab138e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116,

gcloud compute ssh --zone "asia-south1-c" "tsm-sc-012" --project "ucr-ursa-major-lesani-lab" --command "    cd stellar-core;     git pull"
0
Executing: gcloud compute ssh --zone "us-west1-b" "tsm-sc-001" --project "ucr-ursa-major-lesani-lab" --command "    cd /home/tejas;     sudo rm -r stellar-private;     "
Return code for tsm-sc-001: 256
Executing command for tsm-sc-009: gcloud compute scp --zone "asia-south1-c" --project "ucr-ursa-major-lesani-lab"     --recurse "/home/tejas/stellar-private" "tsm-sc-009:/home/tejas/stellar-private"
Command for tsm-sc-009 finished with exit code: 0
Executing: gcloud compute ssh --zone "us-west1-b" "tsm-sc-002" --project "ucr-ursa-major-lesani-lab" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return code for tsm-sc-002: 0
gcloud compute ssh --zone "us-west1-b" "tsm-sc-002" --project "ucr-ursa-major-lesani-lab" --command "    cd stellar-core;     make -j16; cd; sudo rm -r stellar-private"
0
Executing: gcloud com

...

Exception ignored in: <function ResourceTracker.__del__ at 0x734d5f592020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x72a3db982020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


Executing: gcloud compute ssh --zone "us-west1-b" "tsm-sc-005" --project "ucr-ursa-major-lesani-lab" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return code for tsm-sc-005: 256
Executing: gcloud compute ssh --zone "asia-south1-c" "tsm-sc-012" --project "ucr-ursa-major-lesani-lab" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return code for tsm-sc-012: 0
gcloud compute ssh --zone "us-west1-b" "tsm-sc-007" --project "ucr-ursa-major-lesani-lab" --command "    cd stellar-core;     make -j16; cd; sudo rm -r stellar-private"
0
Executing: gcloud compute ssh --zone "asia-south1-c" "tsm-sc-013" --project "ucr-ursa-major-lesani-lab" --command "    cd /home/tejas;     sudo rm -r stellar-private;     "
Return code for tsm-sc-013: 256
Executing: gcloud compute ssh --zone "us-west1-b" "tsm-sc-001" --project "ucr-ursa-major-lesani-lab" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
R

Exception ignored in: <function ResourceTracker.__del__ at 0x73eb1b17e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7a4dfe18e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


Executing: gcloud compute ssh --zone "asia-south1-c" "tsm-sc-014" --project "ucr-ursa-major-lesani-lab" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return code for tsm-sc-014: 256
Executing command to copy node1 from tsm-sc-000: gcloud compute scp --zone "us-west1-b" --project "ucr-ursa-major-lesani-lab"     --recurse "tsm-sc-000:/home/tejas/stellar-private/node1" "/home/tejas/work/experiments/stellar-core/collection_100_rounds_16_zone_3/tsm-sc-000"
Copy from tsm-sc-000 finished with exit code: 0
Executing: gcloud compute ssh --zone "asia-south1-c" "tsm-sc-009" --project "ucr-ursa-major-lesani-lab" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return code for tsm-sc-009: 256
Executing command to copy node3 from tsm-sc-002: gcloud compute scp --zone "us-west1-b" --project "ucr-ursa-major-lesani-lab"     --recurse "tsm-sc-002:/home/tejas/stellar-private/node3" "/home/tejas/work/experiments/stellar-core/collec

Exception ignored in: <function ResourceTracker.__del__ at 0x7a5fd758e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x798c3c57e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ 